# pMoE and MGE: comparing head settings (v2)

**What it does.** Takes a **frozen, pretrained backbone** and trains the two most interesting heads on it, **pMoE** (experts that look at a few image patches) and **MGE** (one expert per backbone depth), each with several settings (the two grids below), so you can see which setting works best.

**How to run**
1. In *Choose the backbone*, write the path of a saved backbone in `BACKBONE_PATH` (for example one from `MoE_comparisons/models/`), or leave it `None` to train a new one (saved under an automatic name in `pMoE_MGE_finetune_v2/models/`). This notebook has its own backbone code and does not need any other notebook.
2. Run the cells from top to bottom. Every run first trains its head on the frozen backbone (phase A), then fine-tunes the backbone's **last stage** together with the head for `FINETUNE_EPOCHS` epochs (phase B, on its own copy: the backbone file is never changed).
3. To look at results again **without training**, set `USE_SAVED = True` (first code cell) and run everything: the backbone, head weights and logs are loaded from disk (or from GitHub on a new pod).
4. Add, remove or change entries of the two grids as you like. To retrain just one setting, leave only that entry in its grid and run again: the comparison at the end also reads the logs of earlier runs on the same backbone from `pMoE_MGE_finetune_v2/history_logs/`.
5. Logs and head weights go to this notebook's own folder, `pMoE_MGE_finetune_v2/`.

Only the **validation** split is used to choose settings; the test split is used once at the end, for the two winners.

In [8]:
USE_SAVED = False   # True: load the saved backbone, head weights and logs instead of training (analyse old runs)

import os, sys, subprocess
import warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')   # take GPU memory as needed, not all of it at start (kernels can then share a GPU)

REPO_URL = 'https://github.com/durak-dev/ImageClassification.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colab starts with an empty VM: clone the repo (code AND data live in it).
    # On later runs the folder already exists, so just pull the newest commits.
    ROOT = '/content/ImageClassification'
    if not os.path.exists(ROOT):
        subprocess.run(['git', 'clone', REPO_URL, ROOT], check=True)
    else:
        # fetch + hard reset instead of `pull`: always lands on the newest origin commit, even if the VM copy diverged
        subprocess.run(['git', '-C', ROOT, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', ROOT, 'reset', '--hard', '@{u}'], check=True)
else:
    # Local: the notebook sits in the project root, but also work if it was
    # started from a subfolder.
    cwd = os.path.abspath(os.getcwd())
    ROOT = cwd if os.path.isdir(os.path.join(cwd, 'utils')) else os.path.dirname(cwd)

os.chdir(ROOT)
sys.path.insert(0, ROOT)          # so `import utils.funcs` finds OUR utils folder

DATA_DIR = os.path.join(ROOT, 'data')
ORIG_DIR = os.path.join(DATA_DIR, 'original')            # raw images/ + labels/ (placed manually, never modified)
MANIFEST_PATH = os.path.join(DATA_DIR, 'split_manifest.csv')   # which file belongs to which split
SEED = 69
NB_DIR = 'pMoE_MGE_finetune_v2'   # this notebook's output folder (relative to the project root)
MODELS_DIR = f'{NB_DIR}/models'          # backbones and head weights
HISTORY_DIR = f'{NB_DIR}/history_logs'   # one CSV log per training run
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)
print('Running on:', 'Google Colab' if IN_COLAB else 'local machine')
print('Project root:', ROOT)

Running on: local machine
Project root: /workspace/ImageClassification


In [9]:
# the kernel survives between runs: drop every cached utils module so the code git just pulled is what gets imported
for _m in [m for m in sys.modules if m == 'utils' or m.startswith('utils.')]:
    del sys.modules[_m]
from utils.funcs import *
from utils import config, github_sync, plotting, resnet
from utils.github_sync import save_csv, load_csv, push_files
from utils.MoE import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import f1_score
tf.keras.mixed_precision.set_global_policy('mixed_float16')

### Where the results go
Logs go to `pMoE_MGE_finetune_v2/history_logs/`, backbones and model weights to `pMoE_MGE_finetune_v2/models/` (this notebook's own output folder). A RunPod pod or Colab VM loses its disk when it ends, so the next cell offers two ways to keep them:
* **Push to GitHub:** paste a GitHub token into the box (hidden, kept in memory only). Everything is then committed to the `checkpoints` branch as soon as it is written, and a new pod downloads the saved backbone from there instead of training it again. Leave the box empty to skip.
* **Copy to your PC:** run `.\pull_results.ps1 -Ip <ip> -Port <port>` before you terminate the pod (see the README).

Running locally you can ignore this cell: everything stays in the project folder.

In [10]:
# Is the disk temporary? Colab VM, or a RunPod pod (pod_bootstrap.sh clones the repo into /workspace).
ON_POD = ROOT.startswith('/workspace/') or bool(os.environ.get('RUNPOD_POD_ID'))
REMOTE = IN_COLAB or ON_POD
if REMOTE:
    config.ensure_github()      # asks once for a GitHub token (hidden box, kept in memory only); empty = no upload
USE_GITHUB = REMOTE and config.github_ready()

if USE_GITHUB:
    _, repo, branch = config.github_settings()
    print(f'Results -> GitHub {repo}@{branch}: every log, backbone and weight file is committed when it is written')
elif ON_POD:
    print('Results stay on the pod. Copy them to your PC before terminating it:  .\\pull_results.ps1 -Ip <ip> -Port <port>')
else:
    print(f'Results -> local folders ({HISTORY_DIR}/, {MODELS_DIR}/)')

Results -> GitHub durak-dev/ImageClassification@checkpoints: every log, backbone and weight file is committed when it is written


In [11]:
# Recreate data/{split}/ with the corrected images + labels if they are missing (fresh Colab VM).
# Needs split_manifest.csv, data/original/ and the two checkpoint files from the cloned repo.
if not all(os.path.exists(os.path.join(split_dir(DATA_DIR, s, 'labels'), CORRECTED_LABELS_FILE)) for s in SPLITS):
    materialize_splits(ORIG_DIR, DATA_DIR, pd.read_csv(MANIFEST_PATH))
    rebuild_corrected(pd.read_csv(os.path.join(ROOT, 'checkpoints', 'df_fixed.csv')), DATA_DIR,
                      labels_corrected=pd.read_csv(os.path.join(ROOT, 'checkpoints', 'labels_corrected.csv')))

In [12]:
dfs = {s: pd.read_csv(os.path.join(split_dir(DATA_DIR, s, "labels"), CORRECTED_LABELS_FILE)) for s in ["train", "val", "test"]}
classes = sorted(dfs["train"]["class_id"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}

## Data
Images are resized to 64x64. **Only the training set is rebalanced** (classes with few images are repeated up to the median class size); validation and test stay as they are.

In [13]:
SIZE, BATCH, AUTOTUNE = 64, 256, tf.data.AUTOTUNE
CLASS_BALANCE = 'oversample'         # train set only: None | "oversample" | "undersample"  (see funcs.balance_frame)
CLASS_BALANCE_TARGET = 'median'  # images per class: None (max / min) | "min" | "median" | "max" | int

def load(path, y):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (SIZE, SIZE))
    return tf.cast(tf.round(img), tf.uint8), y        # uint8: 4x smaller in the RAM cache below

def make_ds(split, shuffle):
    df = dfs[split]
    if split == "train" and CLASS_BALANCE:      # never resample val / test
        df = balance_frame(df, CLASS_BALANCE, CLASS_BALANCE_TARGET, seed=SEED)
        n = df["class_id"].value_counts()
        print(f"train balanced ({CLASS_BALANCE}): {len(df)} images, {n.min()}-{n.max()} per class")
    paths = (f"{DATA_DIR}/{split}/images_corrected/" + df["filename"].astype(str)).values
    labels = df["class_id"].map(class_to_idx)
    assert labels.notna().all(), f"{split} has labels not present in train"
    ds = tf.data.Dataset.from_tensor_slices((paths, labels.values.astype("int32")))
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).cache()   # decode + resize ONCE; later epochs read from RAM
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED)                  # after cache(), so the order still changes every epoch
    ds = ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train, val, test = make_ds("train", True), make_ds("val", False), make_ds("test", False)
print(len(classes), "classes")
print(dfs["train"]["filename"].iloc[0], os.path.exists(f"{DATA_DIR}/train/images_corrected/" + str(dfs["train"]["filename"].iloc[0])))
for x, y in train.take(1):
    print(x.shape, x.dtype, float(tf.reduce_min(x)), float(tf.reduce_max(x)))

train balanced (oversample): 38600 images, 193-193 per class
200 classes
image_000002.JPEG True
(256, 64, 64, 3) <dtype: 'float32'> 0.0 255.0


## Settings
Everything you might want to change is in the next cell and in the two grids further down.

* **Backbone** (only used when `BACKBONE_PATH = None`) - `drop` is the dropout inside the backbone and `weight_decay` the AdamW weight decay.
* **Heads** - every model first trains only its own head on the frozen backbone (`EPOCHS`), then fine-tunes the backbone's last stage together with the head (`FINETUNE_EPOCHS` at `FINETUNE_LR`; `FINETUNE_EPOCHS = 0` switches this off). Training stops early after `PATIENCE` epochs without a better validation loss.

In [20]:
# ======== Settings ========

# --- backbone training (only used when BACKBONE_PATH = None)
BACKBONE = dict(
    blocks=(2, 3, 4),      # residual blocks per stage (was 3, 6, 8); fewer blocks = faster epochs
    drop=0.25,              # dropout inside the backbone (after stages 1 and 2)
    head_drop=0.3,         # dropout of the linear head used only for pretraining
    epochs=40,             # pretraining epochs = length of the cosine schedule
    lr=3e-3,               # PEAK learning rate (linear warmup, then cosine decay to 0)
    warmup_epochs=4,
    weight_decay=0.05,     # AdamW, applied as lr * wd per step (1e-4 was practically none); not on bias / BN
    label_smoothing=0.1,   # ~10% of the labels are wrong: don't let the model be 100% sure
)

# --- heads: phase A trains the head on the frozen backbone, phase B fine-tunes the last backbone stage too
EPOCHS = 20               # phase A: head training epochs (early stopping after PATIENCE epochs without improvement)
PATIENCE = 4              # early stopping: epochs without a better val loss (heads and backbone)
FINETUNE_EPOCHS = 5       # phase B: epochs of fine-tuning the backbone's last stage + head (0 = off)
FINETUNE_LR = 1e-4        # phase B: start learning rate, cosine decay to 0 (no warmup: the head is already trained)
LR = 0.01                 # head PEAK learning rate (warmup + cosine, see HEAD_LR)
USE_AUG = True            # random flip + shift for every model
FEATURE_DROP = 0.4        # CNN / MoE: dropout on the pooled s3 features
N_EXPERTS = 8             # MoE / pMoE: number of experts
BALANCE = 0.1            # MoE: weight of the load-balancing loss
L_PATCHES = 16             # pMoE: patches each expert keeps (at most H x W of s3)
DIVERSITY_WEIGHT = 0.1    # MGE: weight of the diversity penalty

aug = tf.keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomTranslation(0.1, 0.1)])
AUG = aug if USE_AUG else None
STEPS = int(train.cardinality())   # batches per epoch
# every head: 1 epoch linear warmup to LR, then cosine decay to 0 at EPOCHS
HEAD_LR = tf.keras.optimizers.schedules.CosineDecay(0.0, decay_steps=(EPOCHS - 1) * STEPS, warmup_target=LR, warmup_steps=STEPS)


def run(model, name, epochs=EPOCHS, gate=False, initial_epoch=0, save=True):
    """Train `model` and return its History; with USE_SAVED, load its saved weights and log instead.
    Log: <HISTORY_DIR>/<name>.csv, weights (best epoch): <MODELS_DIR>/<name>.weights.h5, both pushed to GitHub if USE_GITHUB.
    initial_epoch > 0 continues an earlier run: its epochs are appended to the same log."""
    log_path, weights_path = f'{HISTORY_DIR}/{name}.csv', f'{MODELS_DIR}/{name}.weights.h5'
    if USE_SAVED:
        for p in (log_path, weights_path):
            if not resnet.fetch_if_missing(p, USE_GITHUB):
                raise FileNotFoundError(f'{p} is missing: train this run first (USE_SAVED = False)')
        model.load_weights(weights_path)
        history = tf.keras.callbacks.History()
        history.history = pd.read_csv(log_path).to_dict('list')
    else:
        callbacks = ([GateMonitor(val)] if gate else []) + [       # before CSVLogger, so their values land in the log
            MacroF1(val),
            tf.keras.callbacks.CSVLogger(log_path, append=initial_epoch > 0),
            tf.keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)]
        history = model.fit(train, validation_data=val, initial_epoch=initial_epoch, epochs=initial_epoch + epochs,
                            callbacks=callbacks)
        files = {log_path: log_path}
        if save:
            model.save_weights(weights_path)
            files[weights_path] = weights_path
        if USE_GITHUB:
            push_files(files, f'run: {name}')
    plot_history(history)
    return history

### Run bookkeeping

In [21]:
import gc
BEST_F1 = {}   # model type -> best val macro-F1 so far (only that run keeps its Keras model)
RUNS = {}   # run name -> dict(model, settings, log (DataFrame), net (Keras model; only kept for the best run of each type, else None))


def unfreeze_last_stage(net):
    """Make the backbone's last stage trainable (every layer after the one that outputs s2); s1, s2 stay frozen."""
    trunk = net.get_layer('trunk')
    trunk.trainable = True
    start = trunk.layers.index(trunk.outputs[1]._keras_history[0]) + 1
    for layer in trunk.layers[:start]:
        layer.trainable = False


def get_run(name, build, model, settings):
    """Phase A: train build()'s head on the frozen backbone. Phase B (FINETUNE_EPOCHS > 0): also fine-tune the
    backbone's last stage at FINETUNE_LR. Both phases go into one log and one weights file.
    With USE_SAVED the saved weights and log are loaded; a run without saved weights is read from its log only."""
    print(f'===== {name}')
    tf.keras.backend.clear_session()
    net = build()
    log_path = f'{HISTORY_DIR}/{name}.csv'
    if USE_SAVED and not resnet.fetch_if_missing(f'{MODELS_DIR}/{name}.weights.h5', USE_GITHUB):
        print(f'{name}: no saved weights, only its log is used')
        if not resnet.fetch_if_missing(log_path, USE_GITHUB):
            raise FileNotFoundError(f'{log_path} is missing too: train this run first (USE_SAVED = False)')
        net = None
    else:
        history = run(net, name)
        if FINETUNE_EPOCHS and not USE_SAVED:
            unfreeze_last_stage(net)
            compile_head(net, tf.keras.optimizers.schedules.CosineDecay(FINETUNE_LR, FINETUNE_EPOCHS * STEPS),
                         BACKBONE['weight_decay'])
            run(net, name, FINETUNE_EPOCHS, initial_epoch=len(history.epoch))
    log = pd.read_csv(log_path)
    # GPU memory: keep only the best model of each type (the winners cells need just those), free the rest
    f1 = log['val_f1'].max()
    if net is not None and f1 > BEST_F1.get(model, -1):
        BEST_F1[model] = f1
        for r in RUNS.values():
            if r['model'] == model:
                r['net'] = None
    else:
        net = None
    gc.collect()
    tf.keras.backend.clear_session()
    RUNS[name] = dict(model=model, settings=settings, log=log, net=net)
    return log, net

## The backbone
A ResNet written for 64x64 images (no early downsampling), written as the function `backbone` in the first code cell below, so you can change it right there (layers, widths; the number of blocks per stage is `BACKBONE['blocks']`). It must return the three feature maps `[s1, s2, s3]` (shallow to deep): `s1` 64x64x32, `s2` 32x32x64, `s3` 16x16x128. pMoE reads `s3`, MGE reads all three.

A saved backbone is a complete `.keras` model (layers and weights), so it can be loaded without this code: editing `backbone` only matters when you train a **new** backbone, and old files keep the layers they were trained with. The cell after the code defines `train_backbone()`; the one after that chooses the backbone.

In [22]:
# The backbone: edit it freely. It must return [s1, s2, s3] (shallow -> deep); the settings come from BACKBONE above.
# Only used when training a NEW backbone: a saved backbone carries its own layers, so editing this cell does not change it.
def backbone(inp):
    """Conv layers only (no augmentation) -> [s1, s2, s3], shallow -> deep."""
    drop, (n1, n2, n3) = BACKBONE['drop'], BACKBONE['blocks']
    x = layers.Rescaling(1 / 255)(inp)
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)     # stem
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # stage 1: 64 x 64 x 32, no downsampling
    for _ in range(n1):
        x = res_block(x, 32, downsample=False)
    s1 = x                                        # MGE expert 0
    x = layers.SpatialDropout2D(drop)(x)
    
    # stage 2: 32 x 32 x 64
    x = res_block(x, 64)
    for _ in range(n2 - 1):
        x = res_block(x, 64, downsample=False)
    s2 = x                                        # MGE expert 1
    x = layers.SpatialDropout2D(drop)(x)

    # stage 3: 16 x 16 x 128
    x = res_block(x, 128)
    for _ in range(n3 - 1):
        x = res_block(x, 128, downsample=False)
    s3 = x                                        # MGE expert 2
    
                                        

    return [s1, s2, s3]

In [23]:
# Train a NEW backbone (only runs when BACKBONE_PATH is None in the cell below).
# It is trained with a plain linear head, then saved as a complete .keras file (layers + weights) in MODELS_DIR under an
# automatic, unique name.
def train_backbone():
    tf.keras.backend.clear_session()
    trunk_in = tf.keras.Input((SIZE, SIZE, 3))
    trunk = tf.keras.Model(trunk_in, backbone(trunk_in), name='trunk')
    path = resnet.new_trunk_path(MODELS_DIR, trunk)
    inp = tf.keras.Input((SIZE, SIZE, 3))
    x = AUG(inp) if AUG is not None else inp
    x = layers.GlobalAveragePooling2D()(trunk(x)[-1])
    x = layers.Dropout(BACKBONE['head_drop'])(x)
    pre = tf.keras.Model(inp, layers.Dense(len(classes))(x), name='pretrain')

    warmup = BACKBONE['warmup_epochs'] * STEPS          # linear warmup to the peak lr, then cosine decay to 0
    lr = tf.keras.optimizers.schedules.CosineDecay(0.0, decay_steps=BACKBONE['epochs'] * STEPS - warmup,
                                                   warmup_target=BACKBONE['lr'], warmup_steps=warmup)
    opt = tf.keras.optimizers.AdamW(lr, weight_decay=BACKBONE['weight_decay'])
    opt.exclude_from_weight_decay(var_names=['bias', 'gamma', 'beta'])   # no decay on biases / batch norm
    pre.compile(opt, sparse_ce(len(classes), BACKBONE['label_smoothing']), metrics=['accuracy'])

    run(pre, os.path.basename(path).removesuffix('.keras') + '_pretrain', epochs=BACKBONE['epochs'], save=False)
    trunk.save(path)                                     # best epoch: EarlyStopping restored the best weights
    print(f'saved {os.path.getsize(path) / 1e6:.1f} MB -> {path}')
    if USE_GITHUB:
        push_files({path: path}, f'backbone: {path}')
    tf.keras.backend.clear_session()
    return path


## Choose the backbone
`BACKBONE_PATH` decides which backbone every run starts from. Each model loads its own copy (`frozen_backbone`): frozen in phase A, last stage fine-tuned in phase B, and the file itself is never changed, so every run starts from exactly the same backbone. It is set to the backbone trained in MoE_comparisons (4 stages, `s3` 8x8x256); with `USE_SAVED = True` it must be a path.

* `BACKBONE_PATH = 'pMoE_MGE_finetune_v2/models/<file>.keras'` (path relative to the project folder) loads that backbone. It can come from either notebook.
* `BACKBONE_PATH = None` trains a new backbone first (the code above), saves it in this notebook's `models/` folder under an automatic name and prints that name, so you can paste it here next time. Files are never overwritten.

The file name goes into the log names, so runs on different backbones never overwrite each other.

In [ ]:
# The backbone every head below starts from. The FILE is never changed by a run.
#   path -> load that backbone, e.g. 'pMoE_MGE_finetune_v2/models/backbone_6072k_20251118-210000.keras' (either notebook's models/)
#   None -> train a NEW backbone now (saved in pMoE_MGE_finetune_v2/models/ under an automatic name, printed below)
BACKBONE_PATH = 'MoE_comparisons/models/backbone_3465k_20261119-173642.keras'

assert BACKBONE_PATH is not None or not USE_SAVED, 'USE_SAVED = True needs BACKBONE_PATH: write the path of the saved backbone'
if BACKBONE_PATH is None:
    BACKBONE_PATH = train_backbone()
    print(f'To reuse it later, write  BACKBONE_PATH = {BACKBONE_PATH!r}')

BACKBONE_PATH = resnet.norm_path(BACKBONE_PATH)
resnet.fetch_if_missing(BACKBONE_PATH, USE_GITHUB)                        # a new pod: download it from GitHub
BACKBONE_ID = os.path.basename(BACKBONE_PATH).removesuffix('.keras')      # part of every log name


def frozen_backbone(inp, augmentation=None):
    """The saved backbone, frozen and in inference mode (batch norm uses its saved statistics): image -> [s1, s2, s3].
    Every model loads its own copy, so no run can change another run's backbone."""
    trunk = tf.keras.models.load_model(BACKBONE_PATH, compile=False)
    trunk.trainable = False
    x = augmentation(inp) if augmentation is not None else inp
    return list(trunk(x, training=False))


_trunk = tf.keras.models.load_model(BACKBONE_PATH, compile=False)
assert _trunk.input_shape[1] == SIZE, f'{BACKBONE_PATH} was trained on {_trunk.input_shape[1]}px images, but SIZE = {SIZE}'
print('using the backbone', BACKBONE_PATH)
print('feature maps (H, W, C):', [tuple(o.shape[1:]) for o in _trunk.outputs])
del _trunk
tf.keras.backend.clear_session()


using the backbone MoE_comparisons/models/backbone_3465k_20261119-173642.keras
feature maps (H, W, C): [(32, 32, 64), (16, 16, 128), (8, 8, 256)]


## Head settings to compare
Each entry is one run; its key is the short name in the logs. The first entry of each grid uses the settings above (`N_EXPERTS`, `L_PATCHES`, `DIVERSITY_WEIGHT`). Add, remove or change entries freely; to retrain just one setting, keep only that entry.

In [27]:
# pMoE: number of experts and patches kept per expert.   MGE: weight of the diversity penalty.
PMOE_GRID = {f'e{n}_l{l}': dict(n_experts=n, l=l) for n, l in [(N_EXPERTS, L_PATCHES), (4, 8), (8, 16)]}
MGE_GRID = {f'div{w}': dict(diversity_weight=w) for w in [DIVERSITY_WEIGHT, 0.3]}
print(list(PMOE_GRID), list(MGE_GRID))

['e8_l16', 'e4_l8'] ['div0.1', 'div0.3']


## pMoE sweep

In [28]:
for key, cfg in PMOE_GRID.items():
    get_run(f'pmoe_{key}_{BACKBONE_ID}',
            lambda: build_pmoe(SIZE, len(classes), augmentation=AUG, backbone_fn=frozen_backbone, lr=HEAD_LR, **cfg),
            'pmoe', key)

===== pmoe_e8_l16_backbone_3465k_20261119-173642


AttributeError: 'list' object has no attribute 'shape'

## MGE sweep

In [ ]:
for key, cfg in MGE_GRID.items():
    get_run(f'mge_{key}_{BACKBONE_ID}',
            lambda: build_mge(SIZE, len(classes), augmentation=AUG, backbone_fn=frozen_backbone, lr=HEAD_LR, **cfg),
            'mge', key)

## Comparison
All choices are made on the **validation** set. For MGE the figures use its `ensemble` output.
Logs of earlier runs on the same backbone are added, so retraining a single setting still shows everything.

In [ ]:
# add the runs on THIS backbone whose logs already exist (earlier sessions, other grids)
for model in ('pmoe', 'mge'):
    for f in sorted(os.listdir(HISTORY_DIR)):
        if f.startswith(f'{model}_') and f.endswith(f'_{BACKBONE_ID}.csv'):
            name = f[:-4]
            if name not in RUNS:
                RUNS[name] = dict(model=model, settings=name[len(model) + 1:-len(BACKBONE_ID) - 1],
                                  log=pd.read_csv(os.path.join(HISTORY_DIR, f)), net=None)


def runs_of(model):
    return {n: r for n, r in RUNS.items() if r['model'] == model}


def label(r):
    return f"{r['model']} {r['settings']}"


def val_acc(log):
    return log['val_ensemble_accuracy'] if 'val_ensemble_accuracy' in log else log['val_accuracy']


summary = pd.DataFrame([dict(run=n, model=r['model'], settings=r['settings'],
                             epochs_run=len(r['log']), best_val_f1=r['log']['val_f1'].max(),
                             at_epoch=int(r['log']['val_f1'].values.argmax()), best_val_acc=val_acc(r['log']).max())
                        for n, r in RUNS.items()]).sort_values('best_val_f1', ascending=False)
best = {m: summary[summary['model'] == m]['run'].iloc[0] for m in ['pmoe', 'mge'] if (summary['model'] == m).any()}
print('best on val:', best)
summary.round(4)

In [ ]:
# pMoE: every head setting
plotting.plot_comparison({label(r): r['log'] for r in runs_of('pmoe').values()})

In [ ]:
# MGE: every head setting
plotting.plot_comparison({label(r): r['log'] for r in runs_of('mge').values()})

In [ ]:
# head to head: the best pMoE vs the best MGE
plotting.plot_comparison({f'{label(RUNS[n])} (best {m})': RUNS[n]['log'] for m, n in best.items()})

## The two winners up close
pMoE: which patches each expert picks (needs the trained model, so not for a run read from its log).
MGE: how good each expert is on its own.

In [ ]:
n = best['pmoe']
net = RUNS[n]['net']
if net is None:
    print(f'{n} was read from its log: train it again in this session to see its patches')
else:
    images, _ = next(iter(val))
    top_idx, gates, grid = expert_choices(net, images[:64])
    print(f'{n}: top_idx {top_idx.shape} = (images, experts, l patches);  patch grid {grid}')
    for i in range(3):
        plotting.show_expert_patches(images[i].numpy(), top_idx[i], grid)

    # do the experts pick different patches? (the diagonal is l; big off-diagonal numbers = no specialisation)
    k = top_idx.shape[1]
    def overlap(i):
        return np.array([[len(set(top_idx[i, a]) & set(top_idx[i, b])) for b in range(k)] for a in range(k)])
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    sns.heatmap(overlap(0), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax[0])
    ax[0].set(title='Shared patches, image 0', xlabel='expert', ylabel='expert')
    sns.heatmap(np.mean([overlap(i) for i in range(len(top_idx))], axis=0), annot=True, fmt='.1f', cmap='Blues',
                cbar=False, ax=ax[1])
    ax[1].set(title=f'Shared patches, mean of {len(top_idx)} images', xlabel='expert', ylabel='expert')
    plt.tight_layout(); plt.show()

In [ ]:
n = best['mge']
h = RUNS[n]['log']
outputs = ['ensemble', 'expert_0', 'expert_1', 'expert_2']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for o in outputs:
    ax[0].plot(h[f'val_{o}_accuracy'].values, marker='o', label=o)
    ax[1].plot(h[f'{o}_accuracy'].values, marker='o', label=o)
ax[0].set(title=f'{label(RUNS[n])}: validation accuracy per output', xlabel='epoch')
ax[1].set(title='training accuracy per output', xlabel='epoch')
ax[0].legend(); plt.tight_layout(); plt.show()
print('best val accuracy:', {o: round(float(h[f'val_{o}_accuracy'].max()), 4) for o in outputs})

## Test set (once, for the two winners)
The settings were chosen on validation; the test set only reports how good the chosen models are.

In [ ]:
y_test = np.concatenate([y.numpy() for _, y in test])
for m, n in best.items():
    net = RUNS[n]['net']
    if net is None:
        print(f'{m}: {n} was read from its log, no model to test (train it again in this session)')
        continue
    pred = net.predict(test, verbose=0)
    if isinstance(pred, (list, tuple)):          # MGE: the first output is the ensemble
        pred = pred[0]
    p = pred.argmax(1)
    print(f'{m:5s} {n}: test macro-F1 {f1_score(y_test, p, average="macro"):.4f}   accuracy {(p == y_test).mean():.4f}')